# ATS-169 / ATS-175: Study 0 â€” Unified Benchmark (Colab GPU)

Run the full Study 0 benchmark on a Colab GPU in a single notebook:

1. **Baselines** â€” XGBoost, Logistic Regression (full + traditional), GBT, FT-Transformer (default)
2. **FT-Transformer hyperparameter sweep** â€” Grid search over lr Ã— n_layers Ã— d_token (36 configs)
3. **SSL Pretraining with tuned architecture** â€” Re-pretrain with best hyperparams at mask ratios 0.15, 0.30, 0.50
4. **Final comparison & Go/No-Go Gate** â€” All models head-to-head with bootstrap CIs

**Gate criterion:** FT-Transformer beats XGBoost on â‰¥3/5 distress outcomes by â‰¥0.01 AUROC.

**Outcomes:** stock_decline, earnings_restate, audit_qualification, sec_enforcement, bankruptcy

**Splits:** train â‰¤2017 | val 2018â€“2019 | test 2020â€“2023

**Prerequisites:**
1. Runtime â†’ Change runtime type â†’ **T4 GPU** (or A100 if available)
2. Data uploaded to Google Drive at `My Drive/Fin-JEPA/data/` with subdirs `raw/` and `processed/`
3. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

In [ ]:
# Shared data loading â€” used by sweep, SSL, and final benchmark cells
from fin_jepa.data.feature_engineering import (
    CATEGORICAL_FEATURES, N_SECTORS, FeatureConfig, build_feature_matrix,
)
from fin_jepa.data.labels import load_label_database
from fin_jepa.data.splits import SplitConfig
from fin_jepa.data.xbrl_loader import load_xbrl_features
from fin_jepa.training.train_study0 import _cfg
from fin_jepa.utils.reproducibility import seed_everything

raw_dir = Path(_cfg(benchmark_cfg, 'data.raw_dir', 'data/raw'))
processed_dir = Path(_cfg(benchmark_cfg, 'data.processed_dir', 'data/processed'))

xbrl_df = load_xbrl_features(raw_dir)
labels_df, _ = load_label_database(processed_dir / 'label_database.parquet')
xbrl_df['period_end'] = pd.to_datetime(xbrl_df['period_end'])
labels_df['period_end'] = pd.to_datetime(labels_df['period_end'])
merged = xbrl_df.merge(labels_df, on=['cik', 'period_end'], how='inner', suffixes=('', '_label'))

split_cfg = SplitConfig(
    train_end=_cfg(benchmark_cfg, 'data.split.train_end', '2017-12-31'),
    val_end=_cfg(benchmark_cfg, 'data.split.val_end', '2019-12-31'),
    test_end=_cfg(benchmark_cfg, 'data.split.test_end', '2023-12-31'),
)
feat_cfg = FeatureConfig(
    use_raw=_cfg(benchmark_cfg, 'features.use_raw', True),
    use_ratios=_cfg(benchmark_cfg, 'features.use_ratios', True),
    use_yoy=_cfg(benchmark_cfg, 'features.use_yoy', True),
    use_sic=_cfg(benchmark_cfg, 'features.use_sic', True),
    use_missingness_flags=_cfg(benchmark_cfg, 'features.use_missingness_flags', True),
    coverage_threshold=_cfg(benchmark_cfg, 'features.coverage_threshold', 0.50),
    normalization_method=_cfg(benchmark_cfg, 'features.normalization_method', 'quantile'),
    median_impute=_cfg(benchmark_cfg, 'features.median_impute', True),
)

universe_df = None
universe_path = raw_dir / 'company_universe.parquet'
if universe_path.exists() and feat_cfg.use_sic:
    universe_df = pd.read_parquet(universe_path)

splits, scaler, feature_cols, categorical_cols = build_feature_matrix(
    merged, split_cfg, feat_cfg, universe_df=universe_df,
)
n_features = len(feature_cols)
n_cat = len(categorical_cols)
cat_cards = [N_SECTORS] * n_cat if n_cat > 0 else None

SEED = int(_cfg(benchmark_cfg, 'training.seed', 42))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Features: {n_features} continuous + {n_cat} categorical')
print(f'Splits: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')
print(f'Device: {device}')

## 2. Baseline Benchmark

Train all baseline models on each distress outcome:
- **XGBoost** (full features)
- **Logistic Regression** â€” full features + traditional ratios only
- **GBT** (raw XBRL features)
- **FT-Transformer** from scratch (default hyperparams)

Also evaluates the built-in go/no-go gate (FT-Transformer vs XGBoost).

In [ ]:
%%time
if not FORCE_RERUN and BENCHMARK_PATH.exists():
    print(f'Loading cached baseline results from {BENCHMARK_PATH}')
    with open(BENCHMARK_PATH) as f:
        benchmark_results = json.load(f)
else:
    from fin_jepa.training.train_study0 import run_benchmark
    benchmark_results = run_benchmark(benchmark_cfg)
    print('Baseline benchmark complete.')

# Normalize format: run_benchmark returns {gate: ..., outcomes: ...}
if 'outcomes' in benchmark_results:
    benchmark_outcomes = benchmark_results['outcomes']
else:
    benchmark_outcomes = benchmark_results  # flat format from older runs

print(f'\nOutcomes evaluated: {list(benchmark_outcomes.keys())}')

In [ ]:
# Baseline results — one styled table per metric (AUROC, AUPRC, Brier, ECE)
MODEL_NAMES = ['xgboost', 'lr_full', 'lr_traditional', 'gbt_raw', 'ft_transformer']
DISPLAY_NAMES = {'xgboost': 'XGBoost', 'lr_full': 'LR (full)', 'lr_traditional': 'LR (trad)',
                 'gbt_raw': 'GBT (raw)', 'ft_transformer': 'FT-Trans'}
METRIC_DISPLAY = {'auroc': 'AUROC', 'auprc': 'AUPRC', 'brier': 'Brier', 'ece': 'ECE'}

def make_metric_df(metric):
    rows = []
    for outcome, models in benchmark_outcomes.items():
        row = {'Outcome': outcome}
        for model in MODEL_NAMES:
            val = (models.get(model) or {}).get(metric)
            row[DISPLAY_NAMES[model]] = val
        rows.append(row)
    return pd.DataFrame(rows).set_index('Outcome')

def highlight_best(s, metric):
    """Bold-green the best value per row; lower is better for Brier/ECE."""
    better_low = metric in ('brier', 'ece')
    valid = s.dropna()
    if valid.empty:
        return ['' for _ in s]
    best_val = valid.min() if better_low else valid.max()
    return ['font-weight: bold; background-color: #d4edda' if v == best_val else '' for v in s]

for metric, label in METRIC_DISPLAY.items():
    df = make_metric_df(metric)
    print(f'\n=== {label} ===')
    display(df.style.apply(highlight_best, metric=metric, axis=1).format('{:.4f}', na_rep='--'))


In [ ]:
# Grouped bar chart: baseline AUROC by outcome
plot_df = baseline_df.dropna(how='all')
if not plot_df.empty:
    ax = plot_df.plot(kind='bar', figsize=(12, 5), width=0.8, edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Baseline Benchmark: Test AUROC by Outcome')
    ax.set_ylim(0.5, max(plot_df.max().max() + 0.05, 0.8))
    ax.legend(loc='upper right', fontsize=9)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='random')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'baseline_auroc_bar.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No baseline results to plot.')


## 3. FT-Transformer Hyperparameter Sweep

Grid search over:
- **Learning rate:** 1e-3, 3e-4, 1e-4, 3e-5
- **n_layers:** 2, 3, 4
- **d_token:** 128, 192, 256

36 configurations, early-stopped on val AUROC. Select best config by
average test AUROC across all evaluable outcomes.

**Expected runtime:** ~10â€“20 min per config on T4 â†’ ~6â€“12 hours total for 36 configs.

**Exit criterion:** Tuned FT-Transformer within 0.02 AUROC of best baseline on each outcome.

In [ ]:
%%time
from itertools import product
from fin_jepa.models.ft_transformer import FTTransformer
from fin_jepa.training.dataset import make_dataloader
from fin_jepa.training.train_study0 import train_ft_transformer, _predict_scores, MIN_POSITIVES
from fin_jepa.training.metrics import compute_all_metrics
import torch.nn as nn

# Sweep grid
SWEEP_LRS = [1e-3, 3e-4, 1e-4, 3e-5]
SWEEP_LAYERS = [2, 3, 4]
SWEEP_DTOKENS = [128, 192, 256]

total = len(SWEEP_LRS) * len(SWEEP_LAYERS) * len(SWEEP_DTOKENS)
print(f'Grid: {len(SWEEP_LRS)} lr x {len(SWEEP_LAYERS)} layers x {len(SWEEP_DTOKENS)} d_token = {total} configs')

if not FORCE_RERUN and SWEEP_PATH.exists():
    print(f'Loading cached sweep results from {SWEEP_PATH}')
    with open(SWEEP_PATH) as f:
        sweep_results = json.load(f)
else:
    sweep_results = {'configs': [], 'best_config': None}
    batch_size = 256
    _cat_cols = categorical_cols or None

    for i, (lr, n_layers, d_token) in enumerate(product(SWEEP_LRS, SWEEP_LAYERS, SWEEP_DTOKENS)):
        # n_heads must divide d_token
        n_heads = 8 if d_token % 8 == 0 else 4
        config_name = f'lr={lr:.0e}_layers={n_layers}_d={d_token}'
        print(f'\n[{i+1}/{total}] {config_name}')

        model_kwargs = {
            'n_features': n_features,
            'd_token': d_token,
            'n_heads': n_heads,
            'n_layers': n_layers,
            'd_ffn_factor': 4,
            'dropout': 0.0,
            'n_outputs': 1,
            'n_cat_features': n_cat,
            'cat_cardinalities': cat_cards,
        }

        config_result = {
            'lr': lr, 'n_layers': n_layers, 'd_token': d_token, 'n_heads': n_heads,
            'outcomes': {},
        }

        for outcome in OUTCOMES:
            train_df = splits['train'][splits['train'][outcome].notna()]
            val_df = splits['val'][splits['val'][outcome].notna()]
            test_df = splits['test'][splits['test'][outcome].notna()]
            n_pos = int(train_df[outcome].sum())
            if n_pos < MIN_POSITIVES:
                config_result['outcomes'][outcome] = {'auroc': None, 'skipped': True}
                print(f'  {outcome}: skipped')
                continue

            n_neg = len(train_df) - n_pos
            pos_weight = n_neg / max(n_pos, 1)

            train_loader = make_dataloader(train_df, feature_cols, outcome, batch_size, shuffle=True, cat_feature_cols=_cat_cols)
            val_loader = make_dataloader(val_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)
            test_loader = make_dataloader(test_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)

            seed_everything(SEED)
            model = FTTransformer(**model_kwargs).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
            criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

            train_ft_transformer(model, train_loader, val_loader, criterion, optimizer,
                                 device, epochs=100, patience=10)

            y_true, y_score = _predict_scores(model, test_loader, device)
            metrics = compute_all_metrics(y_true, y_score)
            config_result['outcomes'][outcome] = metrics
            auroc = metrics.get('auroc')
            print(f'  {outcome}: {auroc:.4f}' if auroc else f'  {outcome}: skipped')

        # Average test AUROC across evaluable outcomes
        aurocs = [m['auroc'] for m in config_result['outcomes'].values()
                  if m.get('auroc') is not None]
        config_result['mean_auroc'] = float(np.mean(aurocs)) if aurocs else 0.0
        sweep_results['configs'].append(config_result)

    # Select best config by mean AUROC
    best = max(sweep_results['configs'], key=lambda c: c['mean_auroc'])
    sweep_results['best_config'] = {
        'lr': best['lr'], 'n_layers': best['n_layers'],
        'd_token': best['d_token'], 'n_heads': best['n_heads'],
        'mean_auroc': best['mean_auroc'],
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(SWEEP_PATH, 'w') as f:
        json.dump(sweep_results, f, indent=2, default=str)
    print(f'\nSweep results saved to {SWEEP_PATH}')

best_cfg = sweep_results['best_config']
print(f"\nBest config: lr={best_cfg['lr']}, n_layers={best_cfg['n_layers']}, "
      f"d_token={best_cfg['d_token']}, mean_auroc={best_cfg['mean_auroc']:.4f}")

In [ ]:
# Sweep results: top 10 configs by mean AUROC
sweep_rows = []
for cfg in sweep_results['configs']:
    row = {'lr': cfg['lr'], 'n_layers': cfg['n_layers'], 'd_token': cfg['d_token'],
           'mean_auroc': cfg['mean_auroc']}
    for outcome in OUTCOMES:
        row[outcome] = cfg['outcomes'].get(outcome, {}).get('auroc')
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows).sort_values('mean_auroc', ascending=False)
sweep_df.head(10).style.format({'lr': '{:.0e}', 'mean_auroc': '{:.4f}',
    **{o: '{:.4f}' for o in OUTCOMES}}, na_rep='--').background_gradient(
    subset=['mean_auroc'], cmap='Greens')

In [ ]:
# Heatmap: mean AUROC by lr x (n_layers, d_token)
pivot_rows = []
for cfg in sweep_results['configs']:
    pivot_rows.append({
        'lr': f"{cfg['lr']:.0e}",
        'arch': f"L={cfg['n_layers']} D={cfg['d_token']}",
        'mean_auroc': cfg['mean_auroc'],
    })

pivot_df = pd.DataFrame(pivot_rows).pivot(index='lr', columns='arch', values='mean_auroc')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_df, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Test AUROC'})
ax.set_title('FT-Transformer Sweep: Mean AUROC by LR x Architecture')
ax.set_ylabel('Learning Rate')
ax.set_xlabel('Architecture (Layers, d_token)')
plt.tight_layout()
fig_dir = RESULTS_DIR / 'figures'
fig_dir.mkdir(exist_ok=True)
plt.savefig(fig_dir / 'sweep_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()


## 4. SSL Pretraining with Tuned Architecture

Re-run SSL pretraining (mask ratios 0.15, 0.30, 0.50) using the best
hyperparameters from the sweep, then fine-tune on all outcomes.

In [ ]:
%%time
if not FORCE_RERUN and SSL_V2_PATH.exists():
    print(f'Loading cached SSL v2 results from {SSL_V2_PATH}')
    with open(SSL_V2_PATH) as f:
        ssl_results = json.load(f)
else:
    # Build tuned SSL config from best sweep params
    best = sweep_results['best_config']
    ssl_overrides = {
        'ft_transformer': {
            'd_token': best['d_token'],
            'n_heads': best['n_heads'],
            'n_layers': best['n_layers'],
        },
        'ssl_experiment': {'force_pretrain': True},
        'checkpoint_dir': 'models/checkpoints/study0_ssl_experiment_v2',
    }
    ssl_cfg_v2 = OmegaConf.merge(ssl_cfg, ssl_overrides)

    from fin_jepa.training.pretrain_ssl import run_ssl_experiment
    ssl_results = run_ssl_experiment(ssl_cfg_v2)

    # Save as v2
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(SSL_V2_PATH, 'w') as f:
        json.dump(ssl_results, f, indent=2, default=str)
    print('SSL v2 experiment complete.')

print(f"\nSSL Recommendation: {ssl_results['recommendation']}")
print(f"Architecture: d_token={best_cfg['d_token']}, n_layers={best_cfg['n_layers']}, "
      f"n_heads={best_cfg['n_heads']}")

In [ ]:
# SSL v2 results: AUROC sweep table + all 4 metrics for scratch vs SSL mr=0.15
mask_ratios = sorted(ssl_results.get('pretrained', {}).keys())
FOUR_METRICS = ['auroc', 'auprc', 'brier', 'ece']
DEFAULT_MR = '0.15'

# AUROC across mask ratios
rows = []
for outcome in OUTCOMES:
    base_rec = (ssl_results.get('baseline') or {}).get(outcome) or {}
    baseline_auroc = base_rec.get('auroc')
    best_auroc, best_mr = baseline_auroc, 'none'
    for mr in mask_ratios:
        auroc = (ssl_results['pretrained'].get(mr) or {}).get(outcome, {}).get('auroc')
        if auroc is not None and (best_auroc is None or auroc > best_auroc):
            best_auroc = auroc
            best_mr = mr
    delta = (best_auroc - baseline_auroc) if baseline_auroc and best_auroc else None
    row = {'Outcome': outcome, 'FT tuned (scratch)': baseline_auroc}
    for mr in mask_ratios:
        row[f'mr={mr}'] = (ssl_results['pretrained'].get(mr) or {}).get(outcome, {}).get('auroc')
    row['Best MR'] = best_mr
    row['AUROC delta'] = delta
    rows.append(row)

ssl_auroc_df = pd.DataFrame(rows).set_index('Outcome')
print('=== AUROC across SSL mask ratios ===')
display(ssl_auroc_df.style.format('{:.4f}', na_rep='--',
    subset=[c for c in ssl_auroc_df.columns if c != 'Best MR']))

# Full 4-metric table: scratch vs SSL mr=DEFAULT_MR
rows2 = []
for outcome in OUTCOMES:
    base_rec = (ssl_results.get('baseline') or {}).get(outcome) or {}
    ssl_rec  = ((ssl_results.get('pretrained') or {}).get(DEFAULT_MR) or {}).get(outcome) or {}
    for metric in FOUR_METRICS:
        rows2.append({
            'Outcome': outcome,
            'Metric': metric.upper(),
            'FT scratch': base_rec.get(metric),
            f'FT SSL mr={DEFAULT_MR}': ssl_rec.get(metric),
        })
ssl_metrics_df = pd.DataFrame(rows2).set_index(['Outcome', 'Metric'])
print(f'\n=== All 4 metrics: scratch vs SSL mr={DEFAULT_MR} ===')
display(ssl_metrics_df.style.format('{:.4f}', na_rep='--'))


In [ ]:
# SSL pretraining loss curves
loss_curves = ssl_results.get('loss_curves', {})

if any(len(v) > 0 for v in loss_curves.values()):
    fig, ax = plt.subplots(figsize=(10, 4))
    for mr, losses in sorted(loss_curves.items()):
        if losses:
            ax.plot(range(1, len(losses) + 1), losses, label=f'mr={mr}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Reconstruction Loss (MSE)')
    ax.set_title('SSL Pretraining Loss Curves (Tuned Architecture)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'ssl_loss_curves.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No loss curves available (loaded from cache).')


## 5. Final Comparison: All Models

Head-to-head on the test set with all contestants:
- LR (traditional ratios)
- LR (full features)
- XGBoost (full features)
- GBT (raw XBRL)
- FT-Transformer (tuned, scratch)
- FT-Transformer (tuned, best SSL pretrained)

In [ ]:
# Unified 6-model comparison — one table per metric, best cell highlighted
ALL_MODEL_KEYS = ['XGBoost', 'LR (full)', 'LR (trad)', 'GBT (raw)', 'FT tuned', 'FT tuned+SSL']
best_sweep = max(sweep_results['configs'], key=lambda c: c['mean_auroc'])

def get_combined_metrics():
    out = {model: {} for model in ALL_MODEL_KEYS}
    for outcome in OUTCOMES:
        bm = benchmark_outcomes.get(outcome, {})
        for src_key, disp in [('xgboost', 'XGBoost'), ('lr_full', 'LR (full)'),
                               ('lr_traditional', 'LR (trad)'), ('gbt_raw', 'GBT (raw)')]:
            rec = bm.get(src_key) or {}
            out[disp][outcome] = {m: rec.get(m) for m in ['auroc', 'auprc', 'brier', 'ece']}
        rec_ft = (best_sweep.get('outcomes') or {}).get(outcome) or {}
        out['FT tuned'][outcome] = {m: rec_ft.get(m) for m in ['auroc', 'auprc', 'brier', 'ece']}
        best_ssl_rec, best_ssl_auroc = {}, -1
        for mr in mask_ratios:
            rec = ((ssl_results.get('pretrained') or {}).get(mr) or {}).get(outcome) or {}
            if rec.get('auroc') is not None and rec['auroc'] > best_ssl_auroc:
                best_ssl_auroc = rec['auroc']
                best_ssl_rec = rec
        out['FT tuned+SSL'][outcome] = {m: best_ssl_rec.get(m) for m in ['auroc', 'auprc', 'brier', 'ece']}
    return out

combined_metrics = get_combined_metrics()

for metric in ['auroc', 'auprc', 'brier', 'ece']:
    rows = []
    for outcome in OUTCOMES:
        row = {'Outcome': outcome}
        for model in ALL_MODEL_KEYS:
            row[model] = (combined_metrics[model].get(outcome) or {}).get(metric)
        rows.append(row)
    df = pd.DataFrame(rows).set_index('Outcome')
    print(f'\n=== {metric.upper()} ===')
    display(df.style.apply(highlight_best, metric=metric, axis=1).format('{:.4f}', na_rep='--'))


In [ ]:
# Hero chart: all models compared
plot_cols = [c for c in ALL_MODELS if c in combined_df.columns]
plot_data = combined_df[plot_cols].dropna(how='all')

if not plot_data.empty:
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']
    ax = plot_data.plot(kind='bar', figsize=(14, 6), width=0.85,
                        color=colors[:len(plot_cols)], edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Study 0 Final Benchmark: All Models (Test Set)')
    ax.set_ylim(0.5, max(plot_data.max().max() + 0.05, 0.8))
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'hero_chart_all_models.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No combined results to plot.')


In [ ]:
# Bootstrap CI — canonical implementation from fin_jepa.training.metrics
from fin_jepa.training.metrics import bootstrap_auroc_ci

def bootstrap_auroc_delta(y_true_ft, y_score_ft, y_true_xgb, y_score_xgb,
                          n_bootstrap=1000, seed=42):
    """Thin backward-compat wrapper around bootstrap_auroc_ci.
    Returns (mean_delta, ci_low, ci_high).
    y_true_ft and y_true_xgb must be the same array (paired test set).
    """
    result = bootstrap_auroc_ci(
        y_true_ft, y_score_ft, y_score_xgb,
        n_bootstrap=n_bootstrap, seed=seed,
    )
    return result['estimate'], result['ci_lower'], result['ci_upper']


In [ ]:
%%time
# Re-generate test predictions: all 6 models x all outcomes x 4 metrics + paired bootstrap CIs
FINAL_RESULT_PATH = RESULTS_DIR / 'final_benchmark.json'

if not FORCE_RERUN and FINAL_RESULT_PATH.exists():
    print(f'Loading cached final benchmark from {FINAL_RESULT_PATH}')
    with open(FINAL_RESULT_PATH) as f:
        final_results = json.load(f)
else:
    import xgboost as xgb
    from fin_jepa.training.metrics import compute_all_metrics
    from fin_jepa.models.baselines import build_logistic_regression, build_gbt
    from fin_jepa.data.feature_engineering import TRADITIONAL_RATIO_FEATURES, RAW_FEATURES

    best = sweep_results['best_config']
    model_kwargs = {
        'n_features': n_features,
        'd_token': best['d_token'],
        'n_heads': best['n_heads'],
        'n_layers': best['n_layers'],
        'd_ffn_factor': 4,
        'dropout': 0.0,
        'n_outputs': 1,
        'n_cat_features': n_cat,
        'cat_cardinalities': cat_cards,
    }

    # Identify best SSL checkpoint
    ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')
    best_mr_key, best_mr_auroc = None, -1
    for mr in mask_ratios:
        aurocs = [ssl_results['pretrained'].get(mr, {}).get(o, {}).get('auroc', 0) for o in OUTCOMES]
        avg = np.mean([a for a in aurocs if a and a > 0]) if any(a and a > 0 for a in aurocs) else 0
        if avg > best_mr_auroc:
            best_mr_auroc = avg
            best_mr_key = mr
    ssl_ckpt_path = ssl_ckpt_dir / f'encoder_mr{best_mr_key}.pt'
    ssl_state_dict = torch.load(ssl_ckpt_path, map_location=device) if ssl_ckpt_path.exists() else None

    _MODEL_KEYS = ['lr_trad', 'lr_full', 'xgboost', 'gbt_raw', 'ft_scratch', 'ft_ssl']
    final_results = {k: {} for k in _MODEL_KEYS}
    # Keep gate_* keys for backward compat with gate-evaluation cells
    final_results.update({'gate_scratch': {}, 'gate_ssl': {}, 'gate_xgb': {}, 'bootstrap': {}})
    batch_size = 256
    _cat_cols = categorical_cols or None
    trad_cols = [c for c in TRADITIONAL_RATIO_FEATURES if c in feature_cols]
    raw_cols  = [c for c in feature_cols if c in RAW_FEATURES]

    for outcome in OUTCOMES:
        test_df  = splits['test'][splits['test'][outcome].notna()]
        train_df = splits['train'][splits['train'][outcome].notna()]
        val_df   = splits['val'][splits['val'][outcome].notna()]
        n_pos = int(train_df[outcome].sum())
        if n_pos < MIN_POSITIVES:
            print(f'{outcome}: skipped (< {MIN_POSITIVES} positives)')
            continue
        n_neg = len(train_df) - n_pos
        pos_weight = n_neg / max(n_pos, 1)

        X_train = np.nan_to_num(train_df[feature_cols].values.astype(np.float32), nan=0.0)
        X_test  = np.nan_to_num(test_df[feature_cols].values.astype(np.float32),  nan=0.0)
        y_train = train_df[outcome].values.astype(np.float32)
        y_test  = test_df[outcome].values.astype(np.float32)

        # LR (full features)
        lr_full_m = build_logistic_regression(C=1.0, max_iter=1000)
        lr_full_m.fit(X_train, y_train)
        final_results['lr_full'][outcome] = compute_all_metrics(
            y_test, lr_full_m.predict_proba(X_test)[:, 1])

        # LR (traditional ratios)
        if trad_cols:
            trad_idx = [feature_cols.index(c) for c in trad_cols]
            lr_trad_m = build_logistic_regression(C=1.0, max_iter=1000)
            lr_trad_m.fit(X_train[:, trad_idx], y_train)
            final_results['lr_trad'][outcome] = compute_all_metrics(
                y_test, lr_trad_m.predict_proba(X_test[:, trad_idx])[:, 1])

        # XGBoost
        xgb_model = xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
            eval_metric='auc', random_state=42, tree_method='hist',
        )
        xgb_model.fit(X_train, y_train)
        xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
        xgb_m = compute_all_metrics(y_test, xgb_scores)
        final_results['xgboost'][outcome] = xgb_m
        final_results['gate_xgb'][outcome] = {'auroc': xgb_m['auroc']}

        # GBT (raw XBRL)
        if raw_cols:
            raw_idx = [feature_cols.index(c) for c in raw_cols]
            gbt_m = build_gbt(max_iter=500, learning_rate=0.05, max_depth=6, min_samples_leaf=20)
            gbt_m.fit(X_train[:, raw_idx], y_train)
            final_results['gbt_raw'][outcome] = compute_all_metrics(
                y_test, gbt_m.predict_proba(X_test[:, raw_idx])[:, 1])

        # FT-Transformer (scratch, tuned)
        train_loader = make_dataloader(train_df, feature_cols, outcome, batch_size,
                                       shuffle=True,  cat_feature_cols=_cat_cols)
        val_loader   = make_dataloader(val_df,   feature_cols, outcome, batch_size,
                                       shuffle=False, cat_feature_cols=_cat_cols)
        test_loader  = make_dataloader(test_df,  feature_cols, outcome, batch_size,
                                       shuffle=False, cat_feature_cols=_cat_cols)

        seed_everything(SEED)
        m_scratch = FTTransformer(**model_kwargs).to(device)
        opt = torch.optim.AdamW(m_scratch.parameters(), lr=best['lr'], weight_decay=1e-5)
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
        train_ft_transformer(m_scratch, train_loader, val_loader, crit, opt,
                             device, epochs=100, patience=10)
        yt_scratch, ys_scratch = _predict_scores(m_scratch, test_loader, device)
        scratch_m = compute_all_metrics(yt_scratch, ys_scratch)
        final_results['ft_scratch'][outcome] = scratch_m
        final_results['gate_scratch'][outcome] = {'auroc': scratch_m['auroc']}

        # FT-Transformer (SSL pretrained)
        if ssl_state_dict is not None:
            seed_everything(SEED)
            m_ssl = FTTransformer(**model_kwargs).to(device)
            m_ssl.load_state_dict(ssl_state_dict, strict=False)
            opt_s  = torch.optim.AdamW(m_ssl.parameters(), lr=best['lr'], weight_decay=1e-5)
            crit_s = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
            train_ft_transformer(m_ssl, train_loader, val_loader, crit_s, opt_s,
                                 device, epochs=100, patience=10)
            yt_ssl, ys_ssl = _predict_scores(m_ssl, test_loader, device)
            ssl_m = compute_all_metrics(yt_ssl, ys_ssl)
            final_results['ft_ssl'][outcome] = ssl_m
            final_results['gate_ssl'][outcome] = {'auroc': ssl_m['auroc']}
            ci_ssl = bootstrap_auroc_delta(yt_ssl, ys_ssl, y_test, xgb_scores)
        else:
            ci_ssl = (None, None, None)

        ci_scratch = bootstrap_auroc_delta(yt_scratch, ys_scratch, y_test, xgb_scores)
        final_results['bootstrap'][outcome] = {
            'scratch_vs_xgb': {'mean': ci_scratch[0], 'ci_low': ci_scratch[1], 'ci_high': ci_scratch[2]},
            'ssl_vs_xgb':     {'mean': ci_ssl[0],     'ci_low': ci_ssl[1],     'ci_high': ci_ssl[2]},
        }
        best_m = (ssl_m if ssl_state_dict else scratch_m)
        print(f"{outcome}: FT_ssl={best_m['auroc']:.4f}  "
              f"XGB={xgb_m['auroc']:.4f}  "
              f"delta={best_m['auroc'] - xgb_m['auroc']:+.4f}  "
              f"95%CI=[{ci_ssl[1] or ci_scratch[1]:+.4f}, {ci_ssl[2] or ci_scratch[2]:+.4f}]")

    final_results['_meta'] = {
        'spec': 'ATS-176',
        'generated_at': pd.Timestamp.now().isoformat(),
        'models': _MODEL_KEYS,
        'metrics': ['auroc', 'auprc', 'brier', 'ece'],
        'ci_method': 'paired_bootstrap',
        'n_bootstrap': 1000,
        'best_ssl_mr': best_mr_key,
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(FINAL_RESULT_PATH, 'w') as f:
        json.dump(final_results, f, indent=2, default=str)
    print(f'\nFinal benchmark saved to {FINAL_RESULT_PATH}')


## 6. Go/No-Go Gate with Bootstrap CIs

**Rule:** FT-Transformer must beat XGBoost on â‰¥3 of 5 outcomes by â‰¥0.01 AUROC on the test set.

We evaluate with:
1. FT-Transformer (tuned, scratch) vs XGBoost
2. FT-Transformer (tuned, best SSL) vs XGBoost

Bootstrap confidence intervals (1000 resamples) on AUROC differences.

In [ ]:
from fin_jepa.training.metrics import go_no_go_gate

In [ ]:
# Go/No-Go Gate
print('=' * 65)
print('GO/NO-GO GATE: FT-Transformer (tuned) vs XGBoost')
print('Rule: FT must beat XGB on >= 3/5 outcomes by >= 0.01 AUROC')
print('=' * 65)

# Use final_results for gate (re-trained with tuned hyperparams)
xgb_gate = final_results.get('gate_xgb', {})
ft_scratch_gate = final_results.get('gate_scratch', {})
ft_ssl_gate = final_results.get('gate_ssl', {})
bootstrap = final_results.get('bootstrap', {})

gate_outcomes_s = [o for o in OUTCOMES if o in xgb_gate and o in ft_scratch_gate
                   and ft_scratch_gate[o].get('auroc') is not None]
gate_outcomes_ssl = [o for o in OUTCOMES if o in xgb_gate and o in ft_ssl_gate
                     and ft_ssl_gate[o].get('auroc') is not None]

print('\n--- Gate 1: FT-Transformer (tuned, scratch) vs XGBoost ---')
if gate_outcomes_s:
    passed_s, wins_s, detail_s = go_no_go_gate(ft_scratch_gate, xgb_gate, gate_outcomes_s)
    status_s = 'PASSED' if passed_s else 'FAILED'
    print(f'Result: {status_s} ({wins_s}/{len(gate_outcomes_s)} wins)\n')
    for o, d in detail_s.items():
        marker = '+' if d['win'] else '-'
        delta = d['ft_auroc'] - d['xgb_auroc']
        bs = bootstrap.get(o, {}).get('scratch_vs_xgb', {})
        ci_str = f"  95%CI=[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]" if bs else ''
        print(f'  [{marker}] {o:25s}  FT={d["ft_auroc"]:.4f}  XGB={d["xgb_auroc"]:.4f}  '
              f'delta={delta:+.4f}{ci_str}')
else:
    print('No overlapping outcomes to evaluate.')

print('\n--- Gate 2: FT-Transformer (tuned, best SSL) vs XGBoost ---')
if gate_outcomes_ssl:
    passed_ssl, wins_ssl, detail_ssl = go_no_go_gate(ft_ssl_gate, xgb_gate, gate_outcomes_ssl)
    status_ssl = 'PASSED' if passed_ssl else 'FAILED'
    print(f'Result: {status_ssl} ({wins_ssl}/{len(gate_outcomes_ssl)} wins)\n')
    for o, d in detail_ssl.items():
        marker = '+' if d['win'] else '-'
        delta = d['ft_auroc'] - d['xgb_auroc']
        bs = bootstrap.get(o, {}).get('ssl_vs_xgb', {})
        ci_str = f"  95%CI=[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]" if bs else ''
        print(f'  [{marker}] {o:25s}  FT={d["ft_auroc"]:.4f}  XGB={d["xgb_auroc"]:.4f}  '
              f'delta={delta:+.4f}{ci_str}')
else:
    print('No overlapping outcomes to evaluate.')

In [ ]:
# Bootstrap CI forest plot
bs_data = final_results.get('bootstrap', {})
plot_outcomes = [o for o in OUTCOMES if o in bs_data and 'scratch_vs_xgb' in bs_data[o]]

if plot_outcomes:
    fig, ax = plt.subplots(figsize=(10, len(plot_outcomes) * 0.8 + 2))
    y_pos = np.arange(len(plot_outcomes))
    width = 0.35

    for i, outcome in enumerate(plot_outcomes):
        # Scratch vs XGB
        bs_s = bs_data[outcome]['scratch_vs_xgb']
        ax.errorbar(bs_s['mean'], i - width/2,
                    xerr=[[bs_s['mean'] - bs_s['ci_low']], [bs_s['ci_high'] - bs_s['mean']]],
                    fmt='o', color='#59a14f', capsize=5, markersize=8,
                    label='FT tuned (scratch)' if i == 0 else '')

        # SSL vs XGB
        if 'ssl_vs_xgb' in bs_data[outcome]:
            bs_ssl = bs_data[outcome]['ssl_vs_xgb']
            ax.errorbar(bs_ssl['mean'], i + width/2,
                        xerr=[[bs_ssl['mean'] - bs_ssl['ci_low']], [bs_ssl['ci_high'] - bs_ssl['mean']]],
                        fmt='s', color='#edc948', capsize=5, markersize=8,
                        label='FT tuned+SSL' if i == 0 else '')

    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
    ax.axvline(x=0.01, color='green', linestyle=':', alpha=0.5, label='Gate margin (0.01)')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_outcomes)
    ax.set_xlabel('AUROC Delta (FT - XGBoost)')
    ax.set_title('Bootstrap 95% CI: FT-Transformer vs XGBoost AUROC Difference')
    ax.legend(loc='lower right')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'bootstrap_ci_forest_plot.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No bootstrap data to plot.')


## 7. Tuned Baselines (Optuna), Multi-Seed & Walk-Forward

### 7a. Optuna-Tuned Baselines

Re-run the baseline benchmark with Optuna temporal-CV tuning enabled (30 trials, 3-fold expanding-window CV). Computational budget matches the FT-Transformer grid search (~30 configs per model).

In [ ]:
%%time
from fin_jepa.training.train_study0 import run_benchmark

# Override config to enable Optuna tuning
tuned_cfg = OmegaConf.to_container(benchmark_cfg, resolve=True)
tuned_cfg['baselines'] = dict(tuned_cfg.get('baselines', {}))
tuned_cfg['baselines']['tune'] = True
tuned_cfg['baselines']['n_trials'] = 30
tuned_cfg['baselines']['n_cv_splits'] = 3
tuned_cfg['results_dir'] = str(RESULTS_DIR)

tuned_baseline_results = run_benchmark(tuned_cfg)

print('Gate:', 'PASSED' if tuned_baseline_results['gate']['passed'] else 'FAILED',
      f'({tuned_baseline_results["gate"]["n_wins"]}/5 wins)')


In [ ]:
# Compare tuned baselines vs. sweep-winning FT-Transformer
import pandas as pd

best = sweep_results['best_config']
rows = []
for oc, res in tuned_baseline_results['outcomes'].items():
    xgb_tuned = res.get('xgboost', {}).get('auroc', float('nan'))
    gbt_tuned = res.get('gbt_raw', {}).get('auroc', float('nan'))
    lr_tuned = res.get('lr_full', {}).get('auroc', float('nan'))
    # Sweep-winning FT from section 3
    ft_sweep = None
    for cfg in sweep_results.get('configs', []):
        if (cfg.get('d_token') == best['d_token'] and
            cfg.get('n_layers') == best['n_layers'] and
            cfg.get('lr') == best['lr']):
            ft_sweep = cfg.get('outcomes', {}).get(oc, {}).get('auroc')
            break
    # SSL FT from section 5 final benchmark
    ft_ssl = None
    try:
        ft_ssl = final_results.get('ft_ssl', {}).get(oc, {}).get('auroc')
    except NameError:
        pass
    rows.append({
        'outcome': oc,
        'XGB (tuned)': round(xgb_tuned, 4) if xgb_tuned == xgb_tuned else None,
        'GBT (tuned)': round(gbt_tuned, 4) if gbt_tuned == gbt_tuned else None,
        'LR (tuned)': round(lr_tuned, 4) if lr_tuned == lr_tuned else None,
        'FT sweep': round(ft_sweep, 4) if ft_sweep else None,
        'FT+SSL': round(ft_ssl, 4) if ft_ssl else None,
    })
tuned_df = pd.DataFrame(rows).set_index('outcome')
display(tuned_df.style.highlight_max(axis=1, color='#c6efce'))

# The real gate comparison: tuned XGB vs sweep/SSL FT-Transformer
print('\n=== Tuned XGB vs. Best FT-Transformer (sweep + SSL) ===')
for idx_row, row in tuned_df.iterrows():
    xgb_v = row.get('XGB (tuned)')
    ft_v = row.get('FT+SSL') or row.get('FT sweep')
    if xgb_v and ft_v:
        delta = ft_v - xgb_v
        tag = 'WIN' if delta >= 0.01 else ('tie' if abs(delta) < 0.01 else 'LOSS')
        print(f'  {idx_row}: XGB={xgb_v:.4f}  FT={ft_v:.4f}  delta={delta:+.4f}  [{tag}]')


### 7b. Multi-Seed FT-Transformer Variance

Train FT-Transformer with 3 different random seeds and report mean ± std AUROC per outcome.
Runs both **scratch** and **best SSL** variants to quantify variance from random initialisation.


In [ ]:
%%time
from fin_jepa.training.train_study0 import run_multiseed_benchmark

multiseed_cfg = OmegaConf.to_container(benchmark_cfg, resolve=True)
multiseed_cfg['results_dir'] = str(RESULTS_DIR)
SEEDS = [42, 123, 456]

# --- Scratch variant ---
multiseed_results = run_multiseed_benchmark(multiseed_cfg, seeds=SEEDS)
print(f'Scratch seeds: {multiseed_results["seeds"]}')

# --- SSL-pretrained variant ---
ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')
best_mr_key = None
if 'ssl_results' in dir() and 'pretrained' in ssl_results:
    mask_ratios_ms = sorted(ssl_results.get('pretrained', {}).keys())
    best_mr_auroc = -1
    for mr in mask_ratios_ms:
        aurocs = [ssl_results['pretrained'].get(mr, {}).get(o, {}).get('auroc', 0)
                  for o in OUTCOMES]
        avg = np.mean([a for a in aurocs if a and a > 0]) if any(a and a > 0 for a in aurocs) else 0
        if avg > best_mr_auroc:
            best_mr_auroc = avg
            best_mr_key = mr

ssl_ckpt_path = ssl_ckpt_dir / f'encoder_mr{best_mr_key}.pt' if best_mr_key else None

multiseed_ssl_results = None
if ssl_ckpt_path and ssl_ckpt_path.exists():
    ssl_ms_cfg = dict(multiseed_cfg)
    ssl_ms_cfg['ssl_checkpoint'] = str(ssl_ckpt_path)
    multiseed_ssl_results = run_multiseed_benchmark(ssl_ms_cfg, seeds=SEEDS)
    print(f'SSL seeds: {multiseed_ssl_results["seeds"]}')
else:
    print('No SSL checkpoint found -- skipping SSL multi-seed variant')


In [ ]:
# Display mean +/- std AUROC per outcome (scratch + SSL)
import pandas as pd

rows = []
for oc, stats in multiseed_results['multiseed'].items():
    row = {
        'outcome': oc,
        'scratch mean': round(stats['mean_auroc'], 4),
        'scratch std': round(stats['std_auroc'], 4),
        'scratch 95% CI': f'{stats["mean_auroc"]:.4f} +/- {2*stats["std_auroc"]:.4f}',
    }
    if multiseed_ssl_results:
        ssl_stats = multiseed_ssl_results['multiseed'].get(oc, {})
        if ssl_stats:
            row['SSL mean'] = round(ssl_stats['mean_auroc'], 4)
            row['SSL std'] = round(ssl_stats['std_auroc'], 4)
            row['SSL 95% CI'] = f'{ssl_stats["mean_auroc"]:.4f} +/- {2*ssl_stats["std_auroc"]:.4f}'
    rows.append(row)
pd.DataFrame(rows).set_index('outcome')


### 7c. Walk-Forward (Expanding-Window) Validation

Expanding-window evaluation across multiple train/test cutoffs (2014–2023). Each fold expands the training set by 1 year and tests on the next 2-year window. Reports XGBoost and FT-Transformer AUROC per fold per outcome.

In [ ]:
%%time
from fin_jepa.training.train_study0 import run_walk_forward

wf_cfg = OmegaConf.to_container(benchmark_cfg, resolve=True)
wf_cfg['results_dir'] = str(RESULTS_DIR)

wf_results = run_walk_forward(wf_cfg)

print(f'Walk-forward: {wf_results["n_folds"]} folds completed')


In [ ]:
# Display walk-forward AUROC table: rows = folds, cols = outcome × model
import pandas as pd
import numpy as np

rows = []
for fold in wf_results["walk_forward_folds"]:
    row = {"fold": fold["label"]}
    for oc, res in fold["outcomes"].items():
        row[f"{oc}|XGB"] = round(res["xgb_auroc"], 4)
        row[f"{oc}|FT"] = round(res["ft_auroc"], 4)
    rows.append(row)
wf_df = pd.DataFrame(rows).set_index("fold")
display(wf_df)

# Summary: mean AUROC across folds per model
ft_cols = [c for c in wf_df.columns if c.endswith("|FT")]
xgb_cols = [c for c in wf_df.columns if c.endswith("|XGB")]
print(f"Mean FT AUROC across folds:  {wf_df[ft_cols].mean().mean():.4f}")
print(f"Mean XGB AUROC across folds: {wf_df[xgb_cols].mean().mean():.4f}")

## 8. Save Results to Drive

In [ ]:
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/Fin-JEPA/results/study0'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Copy all result JSONs
for fname in ['benchmark_results.json', 'ft_sweep_results.json',
              'ssl_experiment_results_v2.json', 'final_benchmark.json',
              'multiseed_results.json', 'walk_forward_results.json']:
    src = RESULTS_DIR / fname
    if src.exists():
        shutil.copy2(str(src), DRIVE_RESULTS)
        print(f'Copied {fname} to Drive')

# Copy figures
fig_dir = RESULTS_DIR / 'figures'
if fig_dir.exists():
    drive_fig_dir = f'{DRIVE_RESULTS}/figures'
    shutil.copytree(str(fig_dir), drive_fig_dir, dirs_exist_ok=True)
    n_figs = len(list(fig_dir.glob('*.png')))
    print(f'Copied {n_figs} figures to Drive')

# Copy SSL v2 checkpoints
for ckpt_name in ['study0_ssl_experiment_v2', 'study0_ssl_experiment']:
    ckpt_src = Path(f'models/checkpoints/{ckpt_name}')
    if ckpt_src.exists():
        drive_ckpt = f'/content/drive/MyDrive/Fin-JEPA/models/checkpoints/{ckpt_name}'
        shutil.copytree(str(ckpt_src), drive_ckpt, dirs_exist_ok=True)
        print(f'Copied {ckpt_name} checkpoints to Drive')

print('\nAll results saved to Google Drive!')
